### Datasets and DataLoaders

In [ ]:
# Instalacion de las librerias necesarias
%pip install transformers
%pip install torch torchvision torchaudio
%pip install opencv-python
%pip install numpy
%pip install albumentations
%pip install pillow
%pip install matplotlib
%pip install pandas

%pip install seaborn
%pip install tqdm

%pip install segmentation_models_pytorch

#%pip install sklearn
%pip install scikit-learn

In [ ]:
# impotacion de librerias
import json, os, torch, cv2, numpy as np, albumentations as A
from PIL import Image; from matplotlib import pyplot as plt
from glob import glob; from PIL import ImageFile
from torch.utils.data import random_split, Dataset, DataLoader
from albumentations.pytorch import ToTensorV2
import random

import sklearn.metrics as metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_auc_score, mean_squared_error, r2_score


import os
import numpy as np

import pandas as pd

import albumentations as A
from albumentations.pytorch import ToTensorV2

ImageFile.LOAD_TRUNCATED_IMAGES = True


import segmentation_models_pytorch as smp, time
from tqdm import tqdm
from torch.nn import functional as F

In [ ]:
# Exploracion del archivo CSV con los labels

path_csv = "dataset\\labels.csv"   # ajusta a tu ruta real
df = pd.read_csv(path_csv)

#limpieza de los nombres de las columnas
df.columns = df.columns.str.strip().str.replace('\ufeff', '')

print(df.head())
print(df.tail())
print(df.shape)

col = "label_list"   # nombre correcto



In [ ]:
for i, row in df.iterrows():
    value = row[col]
    print(f"{i}: {value}")
    
    labels = str(value).split(',')
    if len(labels) > 1:
        print(f"Etiqueta original: {value}")
        new_label = labels[0]
        df.at[i, col] = new_label
        print(f"Etiqueta modificada: {new_label}")


In [ ]:
# Mejor agrupacion de los labels para simplificar el problema de clasificacion

# copia de seguridad del archivo original
df2 = df.copy()
df2.to_csv("dataset/label_list_backup.csv", index=False)

df2 = pd.read_csv("dataset/label_list_backup.csv")

# Diccionario correcto basado en tus 59 labels reales
mapping = {
    # outerwear
    "blazer": "outerwear",
    "jacket": "outerwear",
    "coat": "outerwear",
    "cape": "outerwear",
    "cardigan": "outerwear",
    "suit": "outerwear",

    # topwear
    "t-shirt": "topwear",
    "shirt": "topwear",
    "top": "topwear",
    "sweater": "topwear",
    "hoodie": "topwear",
    "blouse": "topwear",
    "bodysuit": "topwear",
    "sweatshirt": "topwear",
    "vest": "topwear",

    # pants / lowerwear
    "pants": "lowerwear",
    "jeans": "lowerwear",
    "leggings": "lowerwear",
    "skirt": "lowerwear",
    "shorts": "lowerwear",
    "panties": "lowerwear",
    "tights": "lowerwear",

    # dresses
    "dress": "dress",
    "romper": "dress",

    # footwear
    "boots": "footwear",
    "sneakers": "footwear",
    "flats": "footwear",
    "heels": "footwear",
    "loafers": "footwear",
    "sandals": "footwear",
    "wedges": "footwear",
    "pumps": "footwear",
    "clogs": "footwear",
    "shoes": "footwear",

    # big accessories
    "bag": "accessory",
    "purse": "accessory",
    "scarf": "accessory",
    "hat": "accessory",

    # body
    "hair": "body",
    "skin": "body",

    # background
    "background": "background"
}

# Aplicamos el mapeo, si no existe → "ignore"
df2["clean_label"] = df2["label_list"].map(mapping).fillna("ignore")

# Diccionario de mapeo numérico para el modelo
numeric_mapping = {
    "background": 0,
    "topwear": 1,
    "outerwear": 2,
    "lowerwear": 3,
    "dress": 4,
    "footwear": 5,
    "accessory": 6,
    "body": 7,
    "ignore": 255  # para las clases que queremos ignorar
}

# Crear columna con el mapeo numérico
df2["numeric_label"] = df2["clean_label"].map(numeric_mapping)

# Crear columna concatenada con formato "numero: clean_label"
df2["concatenated_label"] = df2.apply(
    lambda row: f"{row['numeric_label']}: {row['clean_label']}", 
    axis=1
)

# Guardar el resultado
df2.to_csv("dataset/label_list_clean_with_numeric.csv", index=False)

print(df2[['Unnamed: 0', 'label_list', 'clean_label', 'numeric_label', 'concatenated_label']].head(20))
print("\nDistribución de las nuevas etiquetas:")
print(df2['numeric_label'].value_counts().sort_index())



In [ ]:
df2

In [ ]:
df2.info()

In [ ]:
df2['numeric_label'].info

In [ ]:
df2

In [ ]:
class CustomSegmentationDataset(Dataset):

    def __init__(self, root, transformations=None, csv_labels_path=None):

        # Paths
        self.im_paths = sorted(glob(os.path.join(root, "png_images", "IMAGES", "*.png")))
        self.gt_paths = sorted(glob(os.path.join(root, "png_masks", "MASKS", "*.png")))

        assert len(self.im_paths) == len(self.gt_paths), \
            "Cantidad de imágenes y máscaras no coincide."

        self.transformations = transformations
        self.ignore_index = 255

        # ============================
        # 1. CSV limpio
        # ============================
        df2 = pd.read_csv(csv_labels_path)
        df2.columns = df2.columns.str.strip().str.replace("\ufeff", "")
        df2["Unnamed: 0"] = df2["Unnamed: 0"].astype(int)
        
        self.id_to_numeric = dict(zip(df2["Unnamed: 0"], df2["numeric_label"]))

        # id original → label limpio
        self.id_to_clean = dict(zip(df2["Unnamed: 0"], df2["clean_label"]))
        
        # Obtener clases válidas (excluyendo ignore)
        valid_classes_df2 = df2[df2["numeric_label"] != self.ignore_index].copy()
        
        # PRIMERO crear label_to_idx
        unique_clean_labels = valid_classes_df2["clean_label"].unique()
        self.label_to_idx = {cls: idx for idx, cls in enumerate(sorted(unique_clean_labels))}
        
        
        # Mapeo inverso para decodificar predicciones
        self.idx_to_label = {idx: cls for cls, idx in self.label_to_idx.items()}
        
        self.n_cls = len(self.label_to_idx)
        
        print("\n=== Información del Dataset ===")
        print(f"Número de clases válidas: {self.n_cls}")
        print(f"Clases y sus índices: {self.label_to_idx}")
        print(f"Índice para ignorar: {self.ignore_index}")
        print(f"Número de imágenes: {len(self.im_paths)}")
        
        

        # Mostrar distribución de clases
        print("\nDistribución de etiquetas originales -> nuevas:")
        for orig_id, num_label in self.id_to_numeric.items():
            clean_label = self.id_to_clean[orig_id]
            print(f"  {orig_id:2d}: {clean_label:15s} → {num_label}")
            
            

    def __len__(self):
        return len(self.im_paths)

    def __getitem__(self, idx):

        im = np.array(Image.open(self.im_paths[idx]).convert("RGB"))
        gt = np.array(Image.open(self.gt_paths[idx]))

        # ============================
        # 2. Remapeo con ignore_index
        # ============================
        remapped = np.full_like(gt, fill_value=self.ignore_index, dtype=np.int64)

        # Aplicar el mapeo directamente
        for original_id, numeric_label in self.id_to_numeric.items():
            remapped[gt == original_id] = numeric_label

        gt = remapped
        
        # Asegurar que gt sea int64
        gt = gt.astype(np.int64)

        # Transformaciones
        if self.transformations:
            augmented = self.transformations(image=im, mask=gt)
            im, gt = augmented["image"], augmented["mask"]
            
        # Asegurar tipo después de transformaciones
        if torch.is_tensor(gt):
            gt = gt.long()  # Convertir a Long
        elif isinstance(gt, np.ndarray):
            gt = gt.astype(np.int64)
            
        # Asegurar que las imágenes estén en float32
        if torch.is_tensor(im):
            im = im.float()
            

        return im, gt
    
    def get_class_names(self):
            """Retorna los nombres de las clases en orden"""
            return [self.idx_to_label[i] for i in range(self.n_cls)]
        
    def get_class_colors(self):
            """Define colores para visualización (opcional)"""
            # Puedes personalizar estos colores
            colors = [
                [0, 0, 0],       # 0: background (negro)
                [255, 0, 0],     # 1: topwear (rojo)
                [0, 255, 0],     # 2: outerwear (verde)
                [0, 0, 255],     # 3: lowerwear (azul)
                [255, 255, 0],   # 4: dress (amarillo)
                [255, 0, 255],   # 5: footwear (magenta)
                [0, 255, 255],   # 6: accessory (cyan)
                [128, 0, 128],   # 7: body (púrpura)
                
            ]
            return colors[:self.n_cls]
    

In [ ]:
def get_dls(
    root,
    transformations,
    bs,
    split=(0.8, 0.1, 0.1),
    ns=2,
    csv_labels_path=None,
    seed=42,
    device="cpu"
):

    ds = CustomSegmentationDataset(
        root=root,
        transformations=transformations,
        csv_labels_path=csv_labels_path
    )
    
        # Obtener información útil para el entrenamiento
    dataset_info = {
        'n_cls': ds.n_cls,  # Número de clases válidas (0-7)
        'ignore_index': ds.ignore_index,  # 255 para píxeles a ignorar
        'class_names': ds.get_class_names(),  # Nombres de clases en orden
        'label_to_idx': ds.label_to_idx,  # Mapeo nombre -> índice
        'idx_to_label': ds.idx_to_label,  # Mapeo índice -> nombre
        'original_labels': ds.id_to_clean  # Mapeo original para referencia
    }
    
    print("=" * 50)
    print("INFORMACIÓN DEL DATASET PARA ENTRENAMIENTO")
    print("=" * 50)
    print(f"Número de clases: {ds.n_cls}")
    print(f"Índice para ignorar: {ds.ignore_index}")
    print(f"Clases: {ds.label_to_idx}")
    print(f"Total muestras: {len(ds)}")
    print("=" * 50)

    # reproducibilidad
    generator = torch.Generator().manual_seed(seed)

    tr_len = int(len(ds) * split[0])
    val_len = int(len(ds) * split[1])
    test_len = len(ds) - tr_len - val_len

    tr_ds, val_ds, test_ds = torch.utils.data.random_split(
        ds, [tr_len, val_len, test_len], generator=generator
    )

    # print(f"\nTrain: {len(tr_ds)}")
    # print(f"Val:   {len(val_ds)}")
    # print(f"Test:  {len(test_ds)}\n")
    
    print(f"\nDIVISIÓN DEL DATASET (seed={seed}):")
    print(f"Train: {len(tr_ds)} muestras ({split[0]*100:.1f}%)")
    print(f"Val:   {len(val_ds)} muestras ({split[1]*100:.1f}%)")
    print(f"Test:  {len(test_ds)} muestras ({split[2]*100:.1f}%)")

    # VERIFICAR DISTRIBUCIÓN DE CLASES EN SPLITS
    # ============================================
    def get_class_distribution(subset):
        """Calcula distribución de clases en un subset"""
        all_masks = []
        for i in range(min(100, len(subset))):  # Muestrear hasta 100
            _, mask = subset[i]
            if torch.is_tensor(mask):
                mask = mask.numpy()
            all_masks.append(mask.flatten())
        
        if all_masks:
            concatenated = np.concatenate(all_masks)
            unique, counts = np.unique(concatenated, return_counts=True)
            dist = dict(zip(unique, counts))
            
            # Convertir índices a nombres de clase
            dist_named = {}
            for idx, count in dist.items():
                if idx == ds.ignore_index:
                    dist_named['ignore'] = count
                elif idx in ds.idx_to_label:
                    dist_named[ds.idx_to_label[idx]] = count
                else:
                    dist_named[str(idx)] = count
            
            return dist_named
        return {}
    
    print("\nDistribución aproximada de clases (primeras 100 muestras):")
    print("Train:", get_class_distribution(tr_ds))
    print("Val:  ", get_class_distribution(val_ds))
    print("Test: ", get_class_distribution(test_ds))


    pin = (device == "cuda")

    tr_dl = DataLoader(tr_ds, batch_size=bs, shuffle=True,
                       num_workers=ns, pin_memory=pin, persistent_workers=(ns > 0))
    val_dl = DataLoader(val_ds, batch_size=bs, shuffle=False,
                        num_workers=ns, pin_memory=pin, persistent_workers=(ns > 0))
    test_dl = DataLoader(test_ds, batch_size=1, shuffle=False,
                         num_workers=ns, pin_memory=pin, persistent_workers=(ns > 0))
    
    print("\n" + "=" * 50)
    print("DATALOADERS CREADOS EXITOSAMENTE")
    print("=" * 50)

    return tr_dl, val_dl, test_dl, ds.n_cls




In [ ]:
root = "dataset"
csv_labels_path = "dataset\\label_list_clean_with_numeric.csv"

# mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
# im_h, im_w = 256, 256


# trans = A.Compose([
#     A.Resize(im_h, im_w),
#     A.Normalize(mean=mean, std=std),
#     ToTensorV2()
# ])

# Definir transformaciones
transform = A.Compose([
        A.Resize(256, 256),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.2),
        A.Normalize(mean=[0.485, 0.456, 0.406], 
                   std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
])

device = "cuda" if torch.cuda.is_available() else "cpu"

tr_dl, val_dl, test_dl, n_cls = get_dls(
    root=root,
    transformations=transform,
    bs=4,
    ns=0,
    csv_labels_path=csv_labels_path,
    device=device
)


## VISUALIZACION DE LA DATA


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import math 
from torchvision import transforms as tfs


# ---------------------------------------------------------
# Invierte normalización ImageNet → devuelve imagen RGB uint8
# ---------------------------------------------------------
def to_numpy_image(t):
    inv = tfs.Compose([
        tfs.Normalize(mean=[0,0,0], std=[1/0.229, 1/0.224, 1/0.225]),
        tfs.Normalize(mean=[-0.485, -0.456, -0.406], std=[1,1,1])
    ])
    t = inv(t)
    t = (t * 255).clamp(0,255).byte().cpu().permute(1,2,0).numpy()
    return t

# ---------------------------------------------------------
# Convierte máscara (tensor HxW con clases) a imagen visible
# ---------------------------------------------------------
def mask_to_rgb(mask, n_cls, ignore_index=255):
    """
    Convierte una máscara HxW (clases) a RGB usando colores fijos
    
    Args:
        mask: Tensor o numpy array de forma (H, W)
        n_cls: Número de clases válidas
        ignore_index: Valor para píxeles ignorados (default: 255)
    
    Returns:
        RGB image de forma (H, W, 3)
    """
    if torch.is_tensor(mask):
        mask = mask.cpu().numpy()
    
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.int64)
    
    
    # Colores fijos para cada clase (más intuitivos que random)
    colors = [
        [0, 0, 0],        # 0: background - negro
        [255, 0, 0],      # 1: topwear - rojo
        [0, 255, 0],      # 2: outerwear - verde
        [0, 0, 255],      # 3: lowerwear - azul
        [255, 255, 0],    # 4: dress - amarillo
        [255, 0, 255],    # 5: footwear - magenta
        [0, 255, 255],    # 6: accessory - cian
        [128, 0, 128],    # 7: body - púrpura
    ]
    
    # Asegurarnos de tener suficientes colores
    colors = colors[:n_cls]
    
    # Mapear cada clase a su color
    for cls_idx in range(n_cls):
        if cls_idx < len(colors):
            rgb[mask == cls_idx] = colors[cls_idx]
    
    # ignore_index → gris oscuro (para distinguir del background)
    rgb[mask == ignore_index] = [64, 64, 64]
    
    return rgb

# ---------------------------------------------------------
# Plot helper
# ---------------------------------------------------------
def plot(ax, image, title=""):
    ax.imshow(image)
    ax.set_title(title)
    ax.axis("off")

# ---------------------------------------------------------
# Versión 1: Visualizar dataset original
# ---------------------------------------------------------
def visualize_dataset(dataset, n_ims=8):
    """
    Visualiza muestras del dataset original (CustomSegmentationDataset)
    """
    import math
    
    total_plots = n_ims * 2
    cols = 4
    rows = math.ceil(total_plots / cols)
    
    fig, axs = plt.subplots(rows, cols, figsize=(20, rows * 4))
    axs = axs.flatten()
    
    # Seleccionar índices aleatorios
    indices = random.sample(range(len(dataset)), n_ims)
    
    i = 0
    for idx in indices:
        im, gt = dataset[idx]
        
        # Convertir imagen y máscara
        rgb = to_numpy_image(im.float())
        gt_img = mask_to_rgb(gt, dataset.n_cls, dataset.ignore_index)
        
        plot(axs[i], rgb, f"Imagen {idx}")
        i += 1
        plot(axs[i], gt_img, f"Máscara (clases {np.unique(gt)})")
        i += 1
        
        if i >= len(axs):
            break
    
    # Ocultar ejes vacíos
    for j in range(i, len(axs)):
        axs[j].axis('off')
    
    plt.suptitle(f"Dataset - {n_cls} clases | ignore_index={dataset.ignore_index}", fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Mostrar estadísticas de las muestras visualizadas
    print("=" * 60)
    print(f"VISUALIZACIÓN - {n_ims} muestras del dataset")
    print("=" * 60)
    for idx in indices:
        im, gt = dataset[idx]
        unique_vals = torch.unique(gt) if torch.is_tensor(gt) else np.unique(gt)
        valid_classes = [v for v in unique_vals if v != dataset.ignore_index and v < dataset.n_cls]
        ignore_pixels = np.sum(gt == dataset.ignore_index) if hasattr(gt, '__len__') else 0
        
        print(f"Muestra {idx}: Clases presentes: {valid_classes}, Píxeles ignorados: {ignore_pixels}")

# ---------------------------------------------------------
# Versión 2: Visualizar Subset (de DataLoader)
# ---------------------------------------------------------
def visualize_subset(subset, n_ims=8):
    """
    Visualiza muestras de un Subset (como tr_dl.dataset, val_dl.dataset)
    """
    import math
    
    total_plots = n_ims * 2
    cols = 4
    rows = math.ceil(total_plots / cols)
    
    fig, axs = plt.subplots(rows, cols, figsize=(20, rows * 4))
    axs = axs.flatten()
    
    # Seleccionar índices aleatorios dentro del subset
    indices = random.sample(range(len(subset)), n_ims)
    
    i = 0
    for plot_idx, subset_idx in enumerate(indices):
        # Obtener muestra del subset
        im, gt = subset[subset_idx]
        
        # Convertir imagen y máscara
        rgb = to_numpy_image(im.float())
        
        # Necesitamos acceder al dataset original para obtener n_cls e ignore_index
        if hasattr(subset, 'dataset') and hasattr(subset.dataset, 'n_cls'):
            gt_img = mask_to_rgb(gt, subset.dataset.n_cls, subset.dataset.ignore_index)
        else:
            # Fallback si no podemos acceder al dataset original
            gt_img = mask_to_rgb(gt, n_cls, 255)
        
        plot(axs[i], rgb, f"Imagen {plot_idx+1}")
        i += 1
        plot(axs[i], gt_img, f"Máscara {plot_idx+1}")
        i += 1
        
        if i >= len(axs):
            break
    
    # Ocultar ejes vacíos
    for j in range(i, len(axs)):
        axs[j].axis('off')
    
    plt.suptitle(f"Subset - {n_cls} clases", fontsize=16)
    plt.tight_layout()
    plt.show()

# ---------------------------------------------------------
# Versión 3: Visualizar DataLoader (batch)
# ---------------------------------------------------------
def visualize_dataloader(dataloader, n_batches=1):
    """
    Visualiza batches completos de un DataLoader
    """
    for batch_idx, (images, masks) in enumerate(dataloader):
        if batch_idx >= n_batches:
            break
            
        batch_size = images.shape[0]
        cols = 4
        rows = math.ceil(batch_size * 2 / cols)
        
        fig, axs = plt.subplots(rows, cols, figsize=(20, rows * 4))
        axs = axs.flatten()
        
        i = 0
        for sample_idx in range(batch_size):
            img = images[sample_idx]
            mask = masks[sample_idx]
            
            rgb = to_numpy_image(img.float())
            
            # Acceder a n_cls desde el dataset
            if hasattr(dataloader.dataset, 'dataset'):
                gt_img = mask_to_rgb(mask, dataloader.dataset.dataset.n_cls, 
                                   dataloader.dataset.dataset.ignore_index)
            else:
                gt_img = mask_to_rgb(mask, n_cls, 255)
            
            plot(axs[i], rgb, f"Batch {batch_idx}, Img {sample_idx}")
            i += 1
            plot(axs[i], gt_img, f"Mask {sample_idx}")
            i += 1
            
            if i >= len(axs):
                break
        
        # Ocultar ejes vacíos
        for j in range(i, len(axs)):
            axs[j].axis('off')
        
        plt.suptitle(f"Batch {batch_idx} - Shape: {images.shape}", fontsize=16)
        plt.tight_layout()
        plt.show()
        
        # Estadísticas del batch
        print(f"\nBatch {batch_idx}:")
        print(f"  Images shape: {images.shape}")
        print(f"  Masks shape: {masks.shape}")
        
        unique_vals = torch.unique(masks)
        print(f"  Unique mask values: {unique_vals.tolist()}")
        
        for val in unique_vals:
            count = (masks == val).sum().item()
            if val == 255:
                print(f"    ignore (255): {count} píxeles")
            elif val < n_cls:
                print(f"    clase {val}: {count} píxeles")

# ---------------------------------------------------------
# USO EN TU NOTEBOOK
# ---------------------------------------------------------

# Opción A: Visualizar el dataset original (antes de split)
print("=" * 60)
print("VISUALIZANDO DATASET ORIGINAL")
print("=" * 60)

# Necesitarías recrear el dataset o guardar una referencia
# ds_original = CustomSegmentationDataset(
#     root=root,
#     transformations=trans,
#     csv_labels_path=csv_labels_path
# )
# visualize_dataset(ds_original, n_ims=8)

# Opción B: Visualizar los subsets (RECOMENDADO)
print("\n" + "=" * 60)
print("VISUALIZANDO SUBSET DE ENTRENAMIENTO")
print("=" * 60)
visualize_subset(tr_dl.dataset, n_ims=8)

print("\n" + "=" * 60)
print("VISUALIZANDO SUBSET DE VALIDACIÓN")
print("=" * 60)
visualize_subset(val_dl.dataset, n_ims=4)

# Opción C: Visualizar un batch del dataloader
print("\n" + "=" * 60)
print("VISUALIZANDO BATCH DEL DATALOADER DE ENTRENAMIENTO")
print("=" * 60)
visualize_dataloader(tr_dl, n_batches=1)

# ---------------------------------------------------------
# Función específica para tu caso actual
# ---------------------------------------------------------
def visualize_current():
    """
    Función específica para visualizar con tu configuración actual
    """
    print("=" * 70)
    print("VISUALIZACIÓN DE DATOS - CONFIGURACIÓN ACTUAL")
    print("=" * 70)
    print(f"Número de clases: {n_cls}")
    print(f"Device: {device}")
    print(f"Batch size train: {tr_dl.batch_size}")
    print(f"Batch size val: {val_dl.batch_size}")
    print(f"Batch size test: {test_dl.batch_size}")
    
    # Verificar acceso a ignore_index
    if hasattr(tr_dl.dataset.dataset, 'ignore_index'):
        ignore_idx = tr_dl.dataset.dataset.ignore_index
        print(f"Ignore index: {ignore_idx}")
    else:
        print("Ignore index: 255 (default)")
        ignore_idx = 255
    
    # Visualizar algunas muestras de entrenamiento
    import math
    
    n_samples = 6
    cols = 4
    rows = math.ceil(n_samples * 2 / cols)
    
    fig, axs = plt.subplots(rows, cols, figsize=(20, rows * 4))
    axs = axs.flatten()
    
    indices = random.sample(range(len(tr_dl.dataset)), n_samples)
    
    i = 0
    for plot_idx, subset_idx in enumerate(indices):
        im, gt = tr_dl.dataset[subset_idx]
        
        rgb = to_numpy_image(im.float())
        gt_img = mask_to_rgb(gt, n_cls, ignore_idx)
        
        # Obtener clases presentes en la máscara
        if torch.is_tensor(gt):
            unique_vals = torch.unique(gt).cpu().numpy()
        else:
            unique_vals = np.unique(gt)
        
        valid_classes = [int(v) for v in unique_vals if v != ignore_idx and v < n_cls]
        
        plot(axs[i], rgb, f"Train Img {plot_idx+1}")
        i += 1
        plot(axs[i], gt_img, f"Mask {plot_idx+1} (cls: {valid_classes})")
        i += 1
        
        if i >= len(axs):
            break
    
    for j in range(i, len(axs)):
        axs[j].axis('off')
    
    plt.suptitle(f'Dataset de Entrenamiento - {n_cls} clases (ignore={ignore_idx})', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Mostrar leyenda de colores
    print("\n" + "=" * 60)
    print("LEGENDA DE COLORES PARA LAS MÁSCARAS")
    print("=" * 60)
    
    colors = [
        ("background", "negro", 0),
        ("topwear", "rojo", 1),
        ("outerwear", "verde", 2),
        ("lowerwear", "azul", 3),
        ("dress", "amarillo", 4),
        ("footwear", "magenta", 5),
        ("accessory", "cian", 6),
        ("body", "púrpura", 7),
    ]
    
    for cls_name, color_name, cls_idx in colors[:n_cls]:
        print(f"  Clase {cls_idx:2d}: {cls_name:15s} → {color_name}")
    print(f"  Ignore (255): gris oscuro")

# Ejecutar visualización
visualize_current()

### Setup de Entrenamiento

In [ ]:
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
import time


class OptimizedSegmentationModel(nn.Module):
    """
    Modelo optimizado para CPU con DeepLabV3Plus
    """
    def __init__(self, n_cls=8, encoder_name="resnet32", encoder_weights="imagenet"):
        super().__init__()
        
        # Usar encoder más ligero para CPU
        # Opciones: "resnet18", "resnet34", "mobilenet_v2", "efficientnet-b0"
        self.model = smp.DeepLabV3Plus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,  # Usar pesos preentrenados
            in_channels=3,
            classes=n_cls,
            activation=None,  # No aplicar activación, la haremos después
        )
        
        print(f"Modelo creado: DeepLabV3Plus con {encoder_name}")
        print(f"Número de parámetros: {self.count_parameters():,}")
        print(f"Clases: {n_cls}")
    
    def forward(self, x):
        return self.model(x)
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# Crear el modelo
n_cls = 8  # Deberías obtener esto de tu dataset
model = OptimizedSegmentationModel(n_cls=n_cls, encoder_name="resnet18")

# Mover a CPU
device = "cpu"
model.to(device)
print(f"Modelo en dispositivo: {device}")


class Metrics():
    
    def __init__(self, pred, gt, loss_fn, n_cls=8, ignore_index=255, eps=1e-8):
        self.pred_logits = pred
        self.pred = torch.argmax(pred, dim=1)  # (batch, H, W)
        
        # Asegurar que gt tenga la forma correcta
        if gt.dim() == 4 and gt.size(1) == 1:
            self.gt = gt.squeeze(1).long()
        elif gt.dim() == 4 and gt.size(1) > 1:
            # Si es one-hot, convertimos
            self.gt = torch.argmax(gt, dim=1).long()
        else:
            self.gt = gt.long()
        
        self.loss_fn = loss_fn
        self.n_cls = n_cls
        self.ignore_index = ignore_index
        self.eps = eps
        
        # Mover todo a CPU para cálculos de métricas
        if self.pred_logits.is_cuda:
            self.pred_logits = self.pred_logits.cpu()
            self.pred = self.pred.cpu()
            self.gt = self.gt.cpu()
    
    def PA(self):
        """Pixel Accuracy (Precisión por píxel)"""
        with torch.no_grad():
            # Filtrar píxeles ignorados
            valid_mask = self.gt != self.ignore_index
            
            if valid_mask.sum().item() == 0:
                return 0.0
            
            correct = (self.pred == self.gt) & valid_mask
            accuracy = correct.sum().float() / valid_mask.sum().float()
            
        return accuracy.item()
    
    def mIoU(self):
        """Mean Intersection over Union"""
        with torch.no_grad():
            # Filtrar píxeles ignorados
            valid_mask = self.gt != self.ignore_index
            
            if valid_mask.sum().item() == 0:
                return 0.0
            
            pred_valid = self.pred[valid_mask]
            gt_valid = self.gt[valid_mask]
            
            ious = []
            
            for cls in range(self.n_cls):
                pred_cls = pred_valid == cls
                gt_cls = gt_valid == cls
                
                intersection = (pred_cls & gt_cls).sum().float().item()
                union = (pred_cls | gt_cls).sum().float().item()
                
                if union > 0:
                    iou = (intersection + self.eps) / (union + self.eps)
                    ious.append(iou)
                else:
                    # Si no hay píxeles de esta clase, omitimos
                    continue
            
            if len(ious) == 0:
                return 0.0
            
            miou = sum(ious) / len(ious)
        
        return miou
    
    def IoU_per_class(self):
        """IoU por clase"""
        with torch.no_grad():
            valid_mask = self.gt != self.ignore_index
            
            if valid_mask.sum().item() == 0:
                return {f"class_{i}": 0.0 for i in range(self.n_cls)}
            
            pred_valid = self.pred[valid_mask]
            gt_valid = self.gt[valid_mask]
            
            ious = {}
            
            for cls in range(self.n_cls):
                pred_cls = pred_valid == cls
                gt_cls = gt_valid == cls
                
                intersection = (pred_cls & gt_cls).sum().float().item()
                union = (pred_cls | gt_cls).sum().float().item()
                
                if union > 0:
                    iou = (intersection + self.eps) / (union + self.eps)
                    ious[f"class_{cls}"] = iou
                else:
                    ious[f"class_{cls}"] = 0.0
        
        return ious
    
    def loss(self):
        return self.loss_fn(self.pred_logits, self.gt)
    


def tic_toc(t0=None):
     if t0 is None:
         return time.time()
     return time.time() - t0


In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss para manejar desbalance de clases
    Útil cuando algunas clases tienen muy pocos píxeles
    """
    def __init__(self, alpha=1, gamma=2, ignore_index=255, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.ignore_index = ignore_index
        self.reduction = reduction
        self.ce_loss = nn.CrossEntropyLoss(
            ignore_index=ignore_index, 
            reduction='none'
        )
    
    def forward(self, pred, target):
        ce_loss = self.ce_loss(pred, target)
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        
        if self.reduction == 'mean':
            # Filtrar píxeles ignorados
            mask = target != self.ignore_index
            if mask.sum() > 0:
                return focal_loss[mask].mean()
            else:
                return torch.tensor(0.0, device=pred.device)
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

### Entrenamiento y validacion

In [ ]:
class CPUTrainer:
    """
    Trainer optimizado para entrenamiento en CPU
    """
    def __init__(self, model, train_loader, val_loader, 
                 criterion, optimizer, scheduler=None,
                 device="cpu", n_cls=8, ignore_index=255):
        
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.n_cls = n_cls
        self.ignore_index = ignore_index
        
        # Historial
        self.history = {
            'train_loss': [],
            'val_loss': [],
            'train_pa': [],
            'val_pa': [],
            'train_miou': [],
            'val_miou': [],
            'lr': []
        }
    
    def train_epoch(self):
        """Entrena una época"""
        self.model.train()
        total_loss = 0.0
        total_pa = 0.0
        total_miou = 0.0
        num_batches = 0
        
        for batch_idx, (images, masks) in enumerate(self.train_loader):
            # Mover a dispositivo
            images = images.to(self.device)
            masks = masks.to(self.device)
            
            # Forward pass
            self.optimizer.zero_grad()
            outputs = self.model(images)
            
            # Calcular pérdida
            loss = self.criterion(outputs, masks)
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping para estabilidad
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            
            self.optimizer.step()
            
            # Calcular métricas
            with torch.no_grad():
                metrics = Metrics(outputs, masks, self.criterion, 
                                 n_cls=self.n_cls, 
                                 ignore_index=self.ignore_index)
                pa = metrics.PA()
                miou = metrics.mIoU()
            
            total_loss += loss.item()
            total_pa += pa
            total_miou += miou
            num_batches += 1
            
            # Log cada cierto número de batches
            if (batch_idx + 1) % 10 == 0:
                print(f"  Batch {batch_idx+1}/{len(self.train_loader)}: "
                      f"Loss: {loss.item():.4f}, PA: {pa:.4f}, mIoU: {miou:.4f}")
        
        return {
            'loss': total_loss / max(num_batches,1),
            'pa': total_pa / max(num_batches,1),
            'miou': total_miou / max(num_batches,1)
        }
    
    def validate(self):
        """Validación"""
        self.model.eval()
        total_loss = 0.0
        total_pa = 0.0
        total_miou = 0.0
        num_batches = 0
        
        with torch.no_grad():
            for images, masks in self.val_loader:
                images = images.to(self.device).float()
                masks = masks.to(self.device).long()
                
                outputs = self.model(images)
                loss = self.criterion(outputs, masks)
                
                metrics = Metrics(outputs, masks, self.criterion,
                                 n_cls=self.n_cls,
                                 ignore_index=self.ignore_index)
                pa = metrics.PA()
                miou = metrics.mIoU()
                
                total_loss += loss.item()
                total_pa += pa
                total_miou += miou
                num_batches += 1
        
        return {
            'loss': total_loss / max(num_batches,1),
            'pa': total_pa / max(num_batches,1),
            'miou': total_miou / max(num_batches,1)
        }
    
    def train(self, num_epochs, save_path="best_model.pth"):
        """Entrenamiento completo"""
        best_miou = 0.0
        
        print("\n" + "="*60)
        print("INICIANDO ENTRENAMIENTO EN CPU")
        print("="*60)
        
        for epoch in range(num_epochs):
            start_time = time.time()
            
            print(f"\nEpoch {epoch+1}/{num_epochs}")
            print("-"*40)
            
            # Entrenar
            train_metrics = self.train_epoch()
            
            # Validar
            val_metrics = self.validate()
            
            # Actualizar scheduler si existe
            if self.scheduler is not None:
                self.scheduler.step(val_metrics['loss'])
            
            # Guardar en historial
            self.history['train_loss'].append(train_metrics['loss'])
            self.history['val_loss'].append(val_metrics['loss'])
            self.history['train_pa'].append(train_metrics['pa'])
            self.history['val_pa'].append(val_metrics['pa'])
            self.history['train_miou'].append(train_metrics['miou'])
            self.history['val_miou'].append(val_metrics['miou'])
            self.history['lr'].append(self.optimizer.param_groups[0]['lr'])
            
            # Tiempo de la época
            epoch_time = time.time() - start_time
            
            # Mostrar resultados
            print(f"\nResumen Epoch {epoch+1}:")
            print(f"  Tiempo: {epoch_time:.2f}s")
            print(f"  Train Loss: {train_metrics['loss']:.4f} | "
                  f"PA: {train_metrics['pa']:.4f} | "
                  f"mIoU: {train_metrics['miou']:.4f}")
            print(f"  Val Loss:   {val_metrics['loss']:.4f} | "
                  f"PA: {val_metrics['pa']:.4f} | "
                  f"mIoU: {val_metrics['miou']:.4f}")
            print(f"  Learning Rate: {self.optimizer.param_groups[0]['lr']:.6f}")
            
            # Guardar mejor modelo
            if val_metrics['miou'] > best_miou:
                best_miou = val_metrics['miou']
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_miou': best_miou,
                    'history': self.history,
                }, save_path)
                print(f"  ✓ Modelo guardado (mIoU: {best_miou:.4f})")
        
        print("\n" + "="*60)
        print("ENTRENAMIENTO COMPLETADO")
        print(f"Mejor mIoU: {best_miou:.4f}")
        print("="*60)
        
        return self.history

In [ ]:
# FUNCIÓN DE ENTRENAMIENTO
# ============================================
def entrenamiento(tr_dl, val_dl, n_cls=8):
    """
    Inicia el entrenamiento del modelo
    
    Args:
        tr_dl: DataLoader de entrenamiento
        val_dl: DataLoader de validación
        n_cls: Número de clases (debería ser 8 según tu mapeo)
    """
    # Configuración
    device = "cpu"
    ignore_index = 255
    num_epochs = 30  # Reducido para prueba, puedes aumentarlo
    learning_rate = 1e-3
    
    print("="*60)
    print("CONFIGURACIÓN DEL ENTRENAMIENTO")
    print("="*60)
    print(f"Dispositivo: {device}")
    print(f"Número de clases: {n_cls}")
    print(f"Índice ignore: {ignore_index}")
    print(f"Épocas: {num_epochs}")
    print(f"Learning rate: {learning_rate}")
    print(f"Tamaño batch entrenamiento: {tr_dl.batch_size}")
    print(f"Tamaño batch validación: {val_dl.batch_size}")
    
    # 1. Crear modelo - USAR RESNET18 EN LUGAR DE RESNET32
    print("\nCreando modelo...")
    model = OptimizedSegmentationModel(
        n_cls=n_cls,
        encoder_name="resnet34",  # ¡CAMBIO IMPORTANTE! resnet18 es más ligero
        encoder_weights="imagenet"
    ).to(device)
    
    # 2. Definir función de pérdida
    print("\nConfigurando función de pérdida...")
    # Prueba con CrossEntropy primero, luego puedes probar FocalLoss
    criterion = nn.CrossEntropyLoss(ignore_index=ignore_index)
    # criterion = FocalLoss(ignore_index=ignore_index)
    
    # 3. Optimizador
    print("Configurando optimizador...")
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )
    
    # 4. Scheduler CORREGIDO (sin verbose o con versión diferente)
    print("Configurando scheduler...")
    try:
        # Intentar con verbose si está disponible
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=3
        )
    except TypeError:
        # Si falla, usar sin verbose
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=3
        )
        print("  Nota: Scheduler sin verbose (versión anterior de PyTorch)")
    
    
    # 5. Crear trainer
    print("\nCreando trainer...")
    trainer = CPUTrainer(
        model=model,
        train_loader=tr_dl,
        val_loader=val_dl,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        n_cls=n_cls,
        ignore_index=ignore_index
    )
    
    # 6. Entrenar
    print("\nIniciando entrenamiento...")
    history = trainer.train(num_epochs=num_epochs, save_path="best_model.pth")
    
    return model, history

In [ ]:
def verificar_dataloaders(tr_dl, val_dl):
    """Verifica que los DataLoaders tengan los tipos de datos correctos"""
    print("="*60)
    print("VERIFICACIÓN DE DATALOADERS")
    print("="*60)
    
    # Verificar un batch de entrenamiento
    images, masks = next(iter(tr_dl))
    
    print("\nBatch de entrenamiento:")
    print(f"  Images type: {images.dtype}, shape: {images.shape}")
    print(f"  Masks type: {masks.dtype}, shape: {masks.shape}")
    print(f"  Images range: [{images.min():.3f}, {images.max():.3f}]")
    print(f"  Masks unique values: {torch.unique(masks)}")
    
    # Verificar que las máscaras sean Long
    if masks.dtype != torch.int64:
        print(f"\n ADVERTENCIA: Máscaras son tipo {masks.dtype}, deberían ser torch.int64 (Long)")
        print("  Convirtiendo a Long...")
        masks = masks.long()
        print(f"  Nuevo tipo: {masks.dtype}")
    
    # Verificar un batch de validación
    images_val, masks_val = next(iter(val_dl))
    
    print("\nBatch de validación:")
    print(f"  Images type: {images_val.dtype}, shape: {images_val.shape}")
    print(f"  Masks type: {masks_val.dtype}, shape: {masks_val.shape}")
    
    # Verificar valores de ignore_index
    ignore_count_train = (masks == 255).sum().item()
    ignore_count_val = (masks_val == 255).sum().item()
    
    print(f"\nPíxeles ignorados (255):")
    print(f"  Train: {ignore_count_train} píxeles")
    print(f"  Val: {ignore_count_val} píxeles")
    
    # Verificar clases presentes
    print("\nClases presentes en batch de train:")
    unique_classes = torch.unique(masks)
    for cls in unique_classes:
        if cls.item() == 255:
            print(f"  Clase 255 (ignore): {(masks == 255).sum().item()} píxeles")
        else:
            print(f"  Clase {cls.item()}: {(masks == cls).sum().item()} píxeles")
    
    return True

# Paso 1: Verificar DataLoaders
print("Paso 1: Verificando DataLoaders...")
verificar_dataloaders(tr_dl, val_dl)

# Paso 2: Crear función de conversión de tipos si es necesario
def fix_dataloader_types(dataloader):
    """Función generadora que convierte máscaras a Long"""
    for images, masks in dataloader:
        yield images, masks.long()


In [ ]:
# FUNCIÓN PARA VISUALIZAR RESULTADOS
# ============================================
def plot_training_history(history):
    """Visualiza el historial de entrenamiento"""
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Pérdida
    axes[0, 0].plot(history['train_loss'], label='Train')
    axes[0, 0].plot(history['val_loss'], label='Val')
    axes[0, 0].set_title('Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Pixel Accuracy
    axes[0, 1].plot(history['train_pa'], label='Train')
    axes[0, 1].plot(history['val_pa'], label='Val')
    axes[0, 1].set_title('Pixel Accuracy')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('PA')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # mIoU
    axes[0, 2].plot(history['train_miou'], label='Train')
    axes[0, 2].plot(history['val_miou'], label='Val')
    axes[0, 2].set_title('Mean IoU')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('mIoU')
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1, 0].plot(history['lr'])
    axes[1, 0].set_title('Learning Rate')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('LR')
    axes[1, 0].grid(True, alpha=0.3)
    
    # Comparación Train vs Val
    axes[1, 1].plot(history['train_loss'], label='Train Loss', alpha=0.7)
    axes[1, 1].plot(history['val_loss'], label='Val Loss', alpha=0.7)
    axes[1, 1].plot(history['train_miou'], label='Train mIoU', alpha=0.7)
    axes[1, 1].plot(history['val_miou'], label='Val mIoU', alpha=0.7)
    axes[1, 1].set_title('Comparación Train vs Val')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    # Mejor época (basado en val_miou)
    best_epoch = np.argmax(history['val_miou'])
    axes[1, 2].text(0.1, 0.5, 
                   f"Mejor época: {best_epoch + 1}\n"
                   f"Mejor mIoU: {history['val_miou'][best_epoch]:.4f}\n"
                   f"Mejor PA: {history['val_pa'][best_epoch]:.4f}\n"
                   f"Loss en mejor época: {history['val_loss'][best_epoch]:.4f}",
                   fontsize=12, 
                   bbox=dict(boxstyle="round,pad=0.3", facecolor="lightblue"))
    axes[1, 2].axis('off')
    
    plt.suptitle('Historial de Entrenamiento', fontsize=16)
    plt.tight_layout()
    plt.show()

# ============================================
# EJECUTAR ENTRENAMIENTO
# ============================================

print("Verificando configuración...")
print(f"tr_dl definido: {'tr_dl' in locals() or 'tr_dl' in globals()}")
print(f"val_dl definido: {'val_dl' in locals() or 'val_dl' in globals()}")
print(f"n_cls: {n_cls if 'n_cls' in locals() or 'n_cls' in globals() else 'No definido'}")

# Si todo está bien, ejecutar entrenamiento
if 'tr_dl' in locals() and 'val_dl' in locals():
    try:
        
        # Iniciar entrenamiento
        model, history = entrenamiento(tr_dl, val_dl, n_cls=n_cls)
        
        # Visualizar resultados
        plot_training_history(history)
        
        print("\n¡Entrenamiento completado exitosamente!")
        print(f"Historial guardado. Mejor mIoU: {max(history['val_miou']):.4f}")
        
    except Exception as e:
        print(f"\nError durante el entrenamiento: {e}")
        
else:
    print("\nError: tr_dl y/o val_dl no están definidos")
    
    

In [ ]:
@torch.no_grad()
def predict_mask(model, image, device):
    """
    image: tensor (3,H,W)
    return: tensor (H,W) con clases predichas
    """
    model.eval()
    image = image.unsqueeze(0).to(device)  # (1,3,H,W)
    outputs = model(image)
    pred = torch.argmax(outputs, dim=1)     # (1,H,W)
    return pred.squeeze(0).cpu()




def visualize_prediction_sample(
    model,
    dataset,
    n_samples=6,
    device="cpu"
):
    """
    Visualiza:
    Imagen | Máscara GT | Máscara Predicha
    """
    import math
    model.eval()

    total_plots = n_samples * 3
    cols = 3
    rows = math.ceil(total_plots / cols)

    fig, axs = plt.subplots(rows, cols, figsize=(18, rows * 4))
    axs = axs.flatten()

    indices = random.sample(range(len(dataset)), n_samples)

    i = 0
    for idx in indices:
        image, gt = dataset[idx]

        # Imagen original
        rgb = to_numpy_image(image.float())

        # GT mask
        gt_rgb = mask_to_rgb(
            gt,
            dataset.dataset.n_cls if hasattr(dataset, "dataset") else n_cls,
            dataset.dataset.ignore_index if hasattr(dataset, "dataset") else 255
        )
        
        # def suppress_background(pred, bg_class=0):
        #     pred = pred.clone()
        #     pred[pred == bg_class] = 255
        #     return pred


        # Predicción
        pred = predict_mask(model, image, device)
        pred_rgb = mask_to_rgb(
            pred,
            dataset.dataset.n_cls if hasattr(dataset, "dataset") else n_cls,
            dataset.dataset.ignore_index if hasattr(dataset, "dataset") else 255
        )

        plot(axs[i], rgb, "Imagen")
        i += 1
        plot(axs[i], gt_rgb, "GT")
        i += 1
        plot(axs[i], pred_rgb, "Predicción")
        i += 1

    for j in range(i, len(axs)):
        axs[j].axis("off")

    plt.suptitle(
        "Visualización de Segmentación: Imagen | GT | Predicción",
        fontsize=16
    )
    plt.tight_layout()
    plt.show()
    
    
    print("=" * 60)
print("VISUALIZANDO PREDICCIONES EN VALIDACIÓN")
print("=" * 60)

visualize_prediction_sample(
    model=model,
    dataset=val_dl.dataset,
    n_samples=6,
    device=device
)



In [ ]:



def flatten_predictions(preds, gts, ignore_index=255):
    """
    Convierte predicciones y ground truth a vectores 1D
    ignorando píxeles con label = ignore_index
    """
    preds = preds.view(-1).cpu().numpy()
    gts   = gts.view(-1).cpu().numpy()

    mask = gts != ignore_index
    return preds[mask], gts[mask]


In [ ]:
all_preds = []
all_gts   = []

preds = torch.argmax(outputs, dim=1)

all_preds.append(preds)
all_gts.append(gts)


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

all_preds = torch.cat(all_preds, dim=0)
all_gts   = torch.cat(all_gts, dim=0)

y_pred, y_true = flatten_predictions(all_preds, all_gts, ignore_index=255)

precision = precision_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

f1 = f1_score(
    y_true, y_pred,
    average="macro",
    zero_division=0
)

print("📊 MÉTRICAS DE CLASIFICACIÓN (PIXEL-WISE)")
print(f"Precision (macro): {precision:.4f}")
print(f"Recall    (macro): {recall:.4f}")
print(f"F1-score  (macro): {f1:.4f}")


In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(n_cls))
)

print("🧩 Matriz de Confusión (pixel-wise)")
print(cm)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=False,
    cmap="Blues",
    xticklabels=range(n_cls),
    yticklabels=range(n_cls)
)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("Confusion Matrix (Pixel-wise)")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure()
plt.plot(epochs, history["train_loss"], label="Train Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss vs Epochs")
plt.legend()
plt.show()


In [ ]:
plt.figure()
plt.plot(epochs, history["val_pa"], label="Validation Pixel Accuracy")
plt.plot(epochs, history["val_miou"], label="Validation mIoU")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Pixel Accuracy & mIoU vs Epochs")
plt.legend()
plt.show()


In [ ]:
plt.figure()
plt.plot(epochs, history["precision"], label="Precision (macro)")
plt.plot(epochs, history["recall"], label="Recall (macro)")
plt.plot(epochs, history["f1"], label="F1-score (macro)")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Classification Metrics vs Epochs")
plt.legend()
plt.show()
